In [ ]:
# ---- CELL R: Rebuild everything from Drive (fresh notebook) ----
from google.colab import drive
drive.mount('/content/drive')
!pip install -q xgboost openpyxl scipy

import os, json, numpy as np, pandas as pd, openpyxl
from scipy.signal import savgol_filter, find_peaks
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

BASE = "/content/drive/MyDrive/CourtVision"
SHOT_LOG_PATH = f"{BASE}/courtvision_shot_log.xlsx"
FEATURES_CSV  = f"{BASE}/shot_features.csv"
SPLIT_CSV     = f"{BASE}/shot_splits.csv"
FULL_LM_DIR   = f"{BASE}/landmarks/full"
MODEL_DIR     = f"{BASE}/models"

SEED, VIS_THRESHOLD = 42, 0.5
MIN_GAP_S, MIN_PROM, MIN_HEIGHT = 2.5, 0.8, 0.0
FROZEN_FRAC, FROZEN_LAG = 0.8, 0.0
NOSE, R_SHOULDER, R_ELBOW, R_WRIST, R_HIP, R_KNEE, R_ANKLE = 0, 12, 14, 16, 24, 26, 28
USED = [NOSE, R_SHOULDER, R_ELBOW, R_WRIST, R_HIP, R_KNEE, R_ANKLE]
FEATURE_NAMES = json.load(open(f"{MODEL_DIR}/xgb_feature_order.json"))

params = dict(n_estimators=200, max_depth=2, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
              min_child_weight=3, reg_lambda=5.0, eval_metric="logloss", random_state=SEED)

def interp_nans(a):
    a = a.copy(); t = np.arange(len(a))
    for c in range(a.shape[1]):
        good = ~np.isnan(a[:, c])
        if good.sum() >= 2:
            a[:, c] = np.interp(t, t[good], a[good, c])
    return a

def smooth(a, win=7):
    n = len(a)
    win = min(win, n if n % 2 == 1 else n - 1)
    if win < 5:
        return a
    return savgol_filter(a, window_length=win, polyorder=2, axis=0)

def angle_at(a, b, c):
    v1, v2 = a - b, c - b
    cosang = (v1 * v2).sum(1) / (np.linalg.norm(v1, axis=1) * np.linalg.norm(v2, axis=1) + 1e-9)
    return np.degrees(np.arccos(np.clip(cosang, -1, 1)))

def lean_from_vertical(p_from, p_to):
    d = p_to - p_from
    return np.degrees(np.arctan2(d[:, 0], d[:, 1]))

def compute_features(lms, fps, release_time):
    T = len(lms)
    feats = {k: np.nan for k in FEATURE_NAMES}
    if T < 10:
        return feats, 0.0
    xy = lms[:, USED, :2].copy()
    vis = lms[:, USED, 2]
    xy[vis < VIS_THRESHOLD] = np.nan
    coverage = float(np.mean(~np.isnan(xy).any(axis=(1, 2))))
    if coverage < 0.3:
        return feats, coverage
    flat = smooth(interp_nans(xy.reshape(T, -1)))
    pts = flat.reshape(T, 7, 2)
    pts[:, :, 1] *= -1
    nose, sh, el, wr, hip, kn, an = [pts[:, i, :] for i in range(7)]
    r = int(np.clip(round(release_time * fps), 0, T - 1))
    lo, hi = max(0, r - 3), min(T, r + 4)
    facing = np.sign(np.nanmedian(wr[lo:hi, 0] - hip[lo:hi, 0]))
    if facing < 0:
        for arr in (nose, sh, el, wr, hip, kn, an):
            arr[:, 0] *= -1
    torso = float(np.median(np.linalg.norm(sh - hip, axis=1))) + 1e-9
    elbow, knee = angle_at(sh, el, wr), angle_at(hip, kn, an)
    hip_ang, sh_ang = angle_at(sh, hip, kn), angle_at(hip, sh, el)
    feats["elbow_angle"], feats["knee_angle"] = elbow[r], knee[r]
    feats["hip_angle"], feats["shoulder_angle"] = hip_ang[r], sh_ang[r]
    feats["wrist_above_shoulder"] = (wr[r, 1] - sh[r, 1]) / torso
    feats["wrist_above_head"] = (wr[r, 1] - nose[r, 1]) / torso
    feats["forearm_lean"] = lean_from_vertical(el, wr)[r]
    feats["trunk_lean"] = lean_from_vertical(hip, sh)[r]
    pre_start = max(0, r - int(1.5 * fps))
    seg = knee[pre_start:r + 1]
    j = int(np.argmin(seg))
    feats["min_knee_pre"] = float(seg[j])
    feats["knee_extension"] = float(knee[r] - seg[j])
    feats["t_min_knee_to_release"] = float((r - (pre_start + j)) / fps)
    away = np.ones(T, dtype=bool)
    away[max(0, r - int(0.6 * fps)):min(T, r + int(0.6 * fps) + 1)] = False
    base = np.median(hip[away, 1]) if away.sum() >= 5 else np.percentile(hip[:, 1], 20)
    jump = (hip[:, 1] - base) / torso
    w0, w1 = max(0, r - int(0.5 * fps)), min(T, r + int(0.5 * fps) + 1)
    feats["jump_at_release"], feats["jump_peak"] = float(jump[r]), float(jump[w0:w1].max())
    sp0 = max(0, r - int(0.8 * fps))
    knee_speed = np.gradient(knee) * fps
    wrist_speed = np.gradient(wr[:, 1]) * fps / torso
    feats["max_knee_ext_speed"] = float(knee_speed[sp0:r + 1].max())
    feats["max_wrist_rise_speed"] = float(wrist_speed[sp0:r + 1].max())
    return feats, coverage

def wrist_and_signal(lms):
    T = len(lms)
    idx = [NOSE, R_SHOULDER, R_WRIST, R_HIP]
    xy = lms[:, idx, :2].copy()
    xy[lms[:, idx, 2] < VIS_THRESHOLD] = np.nan
    pts = smooth(interp_nans(xy.reshape(T, -1))).reshape(T, 4, 2)
    pts[:, :, 1] *= -1
    nose, sh, wr, hip = [pts[:, i] for i in range(4)]
    torso = np.median(np.linalg.norm(sh - hip, axis=1)) + 1e-9
    return wr[:, 1], (wr[:, 1] - nose[:, 1]) / torso

def release_rule(wy, p, fps, frac, lag):
    T = len(wy)
    lo, hi = max(0, p - int(1.5 * fps)), min(T, p + int(0.3 * fps) + 1)
    base = wy[lo:p + 1].min()
    top = wy[max(0, p - int(0.9 * fps)):hi].max()
    ok = np.where(wy[lo:hi] >= base + frac * (top - base))[0]
    k = lo + (ok[0] if len(ok) else p - lo)
    return max(0.0, k / fps - lag)

def find_shots_final(lms, fps):
    wy, sig = wrist_and_signal(lms)
    pk, _ = find_peaks(sig, height=MIN_HEIGHT, prominence=MIN_PROM, distance=int(MIN_GAP_S * fps))
    return [{"peak_t": p / fps, "release_t": release_rule(wy, p, fps, FROZEN_FRAC, FROZEN_LAG)} for p in pk]

# --- tables ---
feat = pd.read_csv(FEATURES_CSV)
spl = pd.read_csv(SPLIT_CSV)[["clip", "split", "fold"]]
df = feat.merge(spl, on="clip", how="left")

ws = openpyxl.load_workbook(SHOT_LOG_PATH, data_only=True).active
ts_map = {}
for video, n, ts, lab in ws.iter_rows(min_row=3, max_row=ws.max_row, max_col=4, values_only=True):
    if video and isinstance(ts, (int, float)):
        ts_map[(str(video).strip(), int(n))] = float(ts)
df["abs_rel"] = [max(0.0, ts_map[(v, int(n))] - 2.0) + r
                 for v, n, r in zip(df["video"], df["shot_number"], df["release_time"])]

test_videos = sorted(df.loc[df["split"] == "test", "video"].unique())
dev_videos = sorted(df.loc[df["split"] == "trainval", "video"].unique())

# --- checks ---
print("rows:", len(df), "| videos:", df["video"].nunique(), "| test videos:", test_videos)
have = set(f for f in os.listdir(FULL_LM_DIR) if f.endswith(".npz"))
need = {os.path.splitext(v)[0] + ".npz" for v in df["video"].unique()}
print("full-video landmark files found:", len(need & have), "of", len(need), "| missing:", sorted(need - have))
print("feature order:", len(FEATURE_NAMES), "features")

Mounted at /content/drive
rows: 115 | videos: 26 | test videos: ['IMG_0862.MOV', 'IMG_0872.MOV', 'IMG_3124.MOV', 'IMG_3126.MOV']
full-video landmark files found: 26 of 26 | missing: []
feature order: 15 features


In [ ]:
# ---- CELL 27: Full-video feature table (app-style) ----
vid_info = df.groupby("video").agg(label=("label", "first"), split=("split", "first")).reset_index()
fold_map = df[df["split"] == "trainval"].groupby("video")["fold"].first().to_dict()

rows = []
for _, vr in vid_info.iterrows():
    v = vr["video"]
    d = np.load(os.path.join(FULL_LM_DIR, os.path.splitext(v)[0] + ".npz"))
    lms, fps = d["lms"], float(d["fps"])
    anchors = df.loc[df["video"] == v, "abs_rel"].values
    for i, s in enumerate(find_shots_final(lms, fps), start=1):
        a = max(0, int(round((s["release_t"] - 2.0) * fps)))
        b = min(len(lms), int(round((s["release_t"] + 2.0) * fps)))
        f, cov = compute_features(lms[a:b], fps, s["release_t"] - a / fps)
        rows.append({"clip": f"{os.path.splitext(v)[0]}_auto{i}", "video": v, "label": vr["label"],
                     "is_3pt": int(vr["label"] == "3pt"), "split": vr["split"],
                     "fold": fold_map.get(v, -1), "release_t": round(s["release_t"], 2),
                     "matches_logged": int(np.any(np.abs(anchors - s["release_t"]) < 2.0)),
                     "coverage": round(cov, 2), **f})

fdf = pd.DataFrame(rows)
fdf.to_csv("/content/drive/MyDrive/CourtVision/shot_features_fullvideo.csv", index=False)

print("Shots found per split:\n", fdf.groupby(["split", "label"]).size().to_string())
print("\nInvented peaks (no logged shot nearby):", int((fdf["matches_logged"] == 0).sum()))
print("Rows with missing features:", int(fdf[FEATURE_NAMES].isna().any(axis=1).sum()))

Shots found per split:
 split     label
test      2pt      10
          3pt      10
trainval  2pt      51
          3pt      45

Invented peaks (no logged shot nearby): 4
Rows with missing features: 0


In [ ]:
# ---- CELL 28: CV on full-video features, same folds, same settings ----
tv = fdf[(fdf["split"] == "trainval") & (fdf["matches_logged"] == 1)].dropna(subset=FEATURE_NAMES).reset_index(drop=True)
Xf, yf = tv[FEATURE_NAMES].values, tv["is_3pt"].values
accs = []
for k in range(5):
    tr, va = np.where(tv["fold"] != k)[0], np.where(tv["fold"] == k)[0]
    m = XGBClassifier(**params).fit(Xf[tr], yf[tr])
    accs.append(accuracy_score(yf[va], m.predict(Xf[va])))
    base = max(yf[va].mean(), 1 - yf[va].mean())
    print(f"Fold {k}: {len(va)} shots | acc {accs[-1]:.2f} | always-guess {base:.2f}")
print(f"\nMean acc {np.mean(accs):.3f} (+/- {np.std(accs):.3f}) on {len(tv)} shots")
print("Earlier: 0.768 with cut-clip features (95 shots); 0.730 when scoring full-video features with the old model")

Fold 0: 22 shots | acc 0.77 | always-guess 0.68
Fold 1: 14 shots | acc 0.93 | always-guess 0.57
Fold 2: 16 shots | acc 0.62 | always-guess 0.56
Fold 3: 20 shots | acc 0.80 | always-guess 0.50
Fold 4: 20 shots | acc 0.80 | always-guess 0.50

Mean acc 0.785 (+/- 0.097) on 92 shots
Earlier: 0.768 with cut-clip features (95 shots); 0.730 when scoring full-video features with the old model


In [ ]:
# ---- CELL 29: FINAL evaluation, full-video features (RUN ONCE) ----
from sklearn.metrics import confusion_matrix, classification_report

te = fdf[(fdf["split"] == "test") & (fdf["matches_logged"] == 1)].dropna(subset=FEATURE_NAMES).reset_index(drop=True)
final2 = XGBClassifier(**params).fit(Xf, yf)                 # all 92 train/val shots
pred = final2.predict(te[FEATURE_NAMES].values)
yt = te["is_3pt"].values

print(f"TEST accuracy: {accuracy_score(yt, pred):.3f} ({int((pred == yt).sum())}/{len(yt)} shots)")
print("\nConfusion matrix (rows = true, cols = predicted; order 2pt, 3pt):")
print(confusion_matrix(yt, pred, labels=[0, 1]))
print("\n", classification_report(yt, pred, target_names=["2pt", "3pt"], digits=2))

te["pred"] = np.where(pred == 1, "3pt", "2pt")
te["correct"] = (te["pred"] == te["label"]).astype(int)
print("Per-video accuracy:\n", te.groupby("video")["correct"].agg(["mean", "count"]).round(2).to_string())

final2.save_model(f"{MODEL_DIR}/xgb_shot_classifier_fullvideo.json")
print("\nSaved ->", f"{MODEL_DIR}/xgb_shot_classifier_fullvideo.json")

TEST accuracy: 0.750 (15/20 shots)

Confusion matrix (rows = true, cols = predicted; order 2pt, 3pt):
[[9 1]
 [4 6]]

               precision    recall  f1-score   support

         2pt       0.69      0.90      0.78        10
         3pt       0.86      0.60      0.71        10

    accuracy                           0.75        20
   macro avg       0.77      0.75      0.74        20
weighted avg       0.77      0.75      0.74        20

Per-video accuracy:
               mean  count
video                    
IMG_0862.MOV   1.0      5
IMG_0872.MOV   0.2      5
IMG_3124.MOV   1.0      5
IMG_3126.MOV   0.8      5

Saved -> /content/drive/MyDrive/CourtVision/models/xgb_shot_classifier_fullvideo.json
